# Two-dimensional dijet unfolding: beam-direction test

Train a flattened $p_{T}^{ave}$–$\eta_{CM}$ response on Pb-going embedding, apply it to the independent p-going embedding reco distribution, and compare the unfolded result with p-going generator truth. The response, training marginals, misses, fakes, and pair classification all come from Pb-going.

In [52]:
%load_ext autoreload
%autoreload 2

import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path('/Users/gnigmat/work/cms/jetAnalysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ROOT_PYTHON_DIR = Path(subprocess.check_output(['root-config', '--libdir'], text=True).strip())
if str(ROOT_PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_PYTHON_DIR))

try:
    import ROOT
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'PyROOT is unavailable. Start Jupyter with py-env/bin/python from the repository root.'
    ) from exc

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import DIJET_DELTA_PHI_SELECTION_LABEL
from hist_analysis.python.histogram_io import resolve_direction_file
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, save_canvas, set_2d_style, set_legend_style,
    set_pad_style, set_unfolding_1d_style,
)

ROOUNFOLD_ROOT = Path(os.environ.get('ROOUNFOLD_ROOT', '/Users/gnigmat/work/RooUnfold'))
for path in (ROOUNFOLD_ROOT / 'src', ROOUNFOLD_ROOT / 'build'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
ROOUNFOLD_LIBRARY = ROOUNFOLD_ROOT / 'build' / 'libRooUnfold.dylib'
if not ROOUNFOLD_LIBRARY.is_file():
    raise FileNotFoundError(
        f'RooUnfold library not found: {ROOUNFOLD_LIBRARY}. '
        'Set ROOUNFOLD_ROOT to the RooUnfold checkout.'
    )

load_status = ROOT.gSystem.Load(str(ROOUNFOLD_LIBRARY))
if load_status < 0:
    raise RuntimeError(f'ROOT failed to load {ROOUNFOLD_LIBRARY}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [53]:
# Set the ROOT style
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetOptTitle(0)
ROOT.gStyle.SetPalette(ROOT.kBird)

In [54]:
# Define useful functions
text = ROOT.TLatex()
text.SetTextFont(42)
text.SetTextSize(0.04)

# Plot CMS header on the canvas
def plotCMSHeader(collSystem=0, energy=8.16):
    # collSystem: 0 = pp, 1 = pPb, 2 = PbPb
    # energy in TeV
    collSystemStr = "pp" if collSystem == 0 else "pPb" if collSystem == 1 else "PbPb"
    t = ROOT.TLatex()
    t.SetTextFont(42)
    t.SetTextSize(0.05)
    t.DrawLatexNDC(0.15, 0.93, "#bf{CMS} #it{Preliminary}")
    t.SetTextSize(0.04)
    t.DrawLatexNDC(0.6, 0.93, f"{collSystemStr} #sqrt{{s_{{NN}}}} = {energy:.2f} TeV")
    t.SetTextSize(0.05)



## Load RooUnfold

In [55]:
try:
    import RooUnfold
except ImportError as exc:
    raise ImportError(
        f"Unable to import RooUnfold after loading {ROOUNFOLD_LIBRARY}. "
        f"Check that ROOUNFOLD_ROOT points to the RooUnfold checkout."
    ) from exc

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
TRAIN_DIRECTION = 'Pbgoing'  # response-training embedding direction
TEST_DIRECTION = 'pgoing'    # independent embedding direction to unfold
FILE_STEM = 'jetId'
# List of eta cuts for analysis
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
# Finite pT-average intervals used for the 2D unfolding. Values outside this range are excluded.
PT_AVE_BINS = (0, 40, 80, 180, 250, 300, 500, 1000)
ETA_CUT_INDEX = 5
N_ITERATIONS = 20
MEASURED_HISTOGRAM_TEMPLATE = 'hRecoDijetPtEtaCMJerDefExtraUnfold_{eta_cut_index}'
MEASURED_LABEL = 'Reco JER def.+#eta-dep.'
RESPONSE_HISTOGRAM_TEMPLATE = 'hGenDijetPtEtaCMVsRecoJerDefExtraPtEtaCM_{eta_cut_index}'
MISS_HISTOGRAM_TEMPLATE = 'hGenDijetPtEtaCMMissJerDefExtra_{eta_cut_index}'
FAKE_HISTOGRAM_TEMPLATE = 'hRecoDijetPtEtaCMFakeJerDefExtra_{eta_cut_index}'
CLASSIFICATION_HISTOGRAM_TEMPLATE = 'hUnfoldingPairClassificationJerDefExtra_{eta_cut_index}'
# Keep weighted response entries above RooUnfold's internal 1e-9 sanitization threshold.
RESPONSE_SCALE = 1.0e12
FLATTENED_RATIO_TO_GEN_Y_RANGE = (0.5, 2.5)
ETA_RATIO_TO_GEN_Y_RANGE = (0.75, 1.25)
DRAW_GRID = True
SAVE_PNG = False

if GENERATOR not in ('embedding', 'pythia'):
    raise ValueError(f'Unsupported GENERATOR={GENERATOR!r}')
for direction_name, direction in (('TRAIN_DIRECTION', TRAIN_DIRECTION), ('TEST_DIRECTION', TEST_DIRECTION)):
    if direction not in ('pgoing', 'Pbgoing'):
        raise ValueError(f'Unsupported {direction_name}={direction!r}')
if TRAIN_DIRECTION == TEST_DIRECTION:
    raise ValueError('TRAIN_DIRECTION and TEST_DIRECTION must be different for this test')
if RESPONSE_SCALE <= 0.0:
    raise ValueError(f'RESPONSE_SCALE must be positive, got {RESPONSE_SCALE}')
for range_name, y_range in (
    ('FLATTENED_RATIO_TO_GEN_Y_RANGE', FLATTENED_RATIO_TO_GEN_Y_RANGE),
    ('ETA_RATIO_TO_GEN_Y_RANGE', ETA_RATIO_TO_GEN_Y_RANGE),
):
    if len(y_range) != 2 or y_range[0] >= y_range[1]:
        raise ValueError(f'{range_name} must be an increasing (minimum, maximum) pair')

generator_label = GENERATOR.capitalize()
training_label = f'{generator_label} {TRAIN_DIRECTION} response'
test_label = f'{generator_label} {TEST_DIRECTION} test'
train_input_path = resolve_direction_file(BASE_DIR, GENERATOR, TRAIN_DIRECTION, FILE_STEM)
test_input_path = resolve_direction_file(BASE_DIR, GENERATOR, TEST_DIRECTION, FILE_STEM)
eta_cut_tag = f'{ETA_CUTS[ETA_CUT_INDEX]:g}'.replace('.', 'p')
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_UNFOLD2D_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'unfold2D',
))
OUTPUT_TAG = (f'{GENERATOR}_{TRAIN_DIRECTION}_response_{TEST_DIRECTION}_test_'
              f'unfold2D_jerDefExtra_eta_{eta_cut_tag}_iter_{N_ITERATIONS}')
OUTPUT_ROOT_FILE = OUTPUT_DIR / f'{OUTPUT_TAG}.root'
eta_cuts = ETA_CUTS
pt_ave_bins = PT_AVE_BINS

generator_label

'Embedding'

In [57]:
train_input_file = ROOT.TFile.Open(str(train_input_path))
if not train_input_file or train_input_file.IsZombie():
    raise FileNotFoundError(f"Unable to open training ROOT file: {train_input_path}")
test_input_file = ROOT.TFile.Open(str(test_input_path))
if not test_input_file or test_input_file.IsZombie():
    train_input_file.Close()
    raise FileNotFoundError(f"Unable to open test ROOT file: {test_input_path}")

n_eta_cuts = len(eta_cuts)

genPtEtaCM = []
recoPtEtaCM = []
trainingGenPtEtaCM = []
trainingRecoPtEtaCM = []
genPtEtaCMMiss = []
recoPtEtaCMFake = []
genPtEtaCMVsRecoPtEtaCM = []
unfoldingPairClassification = []

for i in range(n_eta_cuts):
    # Gen (truth) distribution
    # Independent test distributions: unfold p-going reco and compare to p-going gen.
    gen_hist = test_input_file.Get(f"hGenDijetPtEtaCM_{i}")
    reco_histogram_key = MEASURED_HISTOGRAM_TEMPLATE.format(eta_cut_index=i)
    reco_hist = test_input_file.Get(reco_histogram_key)
    # Pb-going training marginals and all response-model inputs.
    training_gen_hist = train_input_file.Get(f"hGenDijetPtEtaCM_{i}")
    training_reco_hist = train_input_file.Get(reco_histogram_key)
    response_histogram_key = RESPONSE_HISTOGRAM_TEMPLATE.format(eta_cut_index=i)
    miss_histogram_key = MISS_HISTOGRAM_TEMPLATE.format(eta_cut_index=i)
    fake_histogram_key = FAKE_HISTOGRAM_TEMPLATE.format(eta_cut_index=i)
    classification_histogram_key = CLASSIFICATION_HISTOGRAM_TEMPLATE.format(eta_cut_index=i)
    miss_hist = train_input_file.Get(miss_histogram_key)
    fake_hist = train_input_file.Get(fake_histogram_key)
    response_source = train_input_file.Get(response_histogram_key)
    classification_hist = train_input_file.Get(classification_histogram_key)

    # Report if any of histograms are missing
    if gen_hist is None:
        raise KeyError(f"Missing histogram hGenDijetPtEtaCM_{i} in {test_input_path}")
    if reco_hist is None:
        raise KeyError(f"Missing histogram {reco_histogram_key} in {test_input_path}")
    if training_gen_hist is None:
        raise KeyError(f"Missing histogram hGenDijetPtEtaCM_{i} in {train_input_path}")
    if training_reco_hist is None:
        raise KeyError(f"Missing histogram {reco_histogram_key} in {train_input_path}")
    if miss_hist is None:
        raise KeyError(f"Missing histogram {miss_histogram_key} in {train_input_path}")
    if fake_hist is None:
        raise KeyError(f"Missing histogram {fake_histogram_key} in {train_input_path}")
    if response_source is None:
        raise KeyError(f"Missing histogram {response_histogram_key} in {train_input_path}")
    if classification_hist is None:
        raise KeyError(f"Missing histogram {classification_histogram_key} in {train_input_path}")
    response_hist = response_source.Clone(response_histogram_key)
    
    gen_hist.SetDirectory(0)
    reco_hist.SetDirectory(0)
    training_gen_hist.SetDirectory(0)
    training_reco_hist.SetDirectory(0)
    miss_hist.SetDirectory(0)
    fake_hist.SetDirectory(0)
    classification_hist.SetDirectory(0)
    # response_hist.SetDirectory(0)

    genPtEtaCM.append(gen_hist)
    recoPtEtaCM.append(reco_hist)
    trainingGenPtEtaCM.append(training_gen_hist)
    trainingRecoPtEtaCM.append(training_reco_hist)
    genPtEtaCMMiss.append(miss_hist)
    recoPtEtaCMFake.append(fake_hist)
    genPtEtaCMVsRecoPtEtaCM.append(response_hist)
    unfoldingPairClassification.append(classification_hist)

train_input_file.Close()
test_input_file.Close()

In [58]:
# Demonstrate the unfolding pair classification for the configured eta cut
classification = unfoldingPairClassification[ETA_CUT_INDEX].Clone(
    f'hUnfoldingPairClassificationJerDefExtra_demo_{ETA_CUT_INDEX}'
)
classification.SetDirectory(0)
classification_integral = classification.Integral()
if classification_integral <= 0.0:
    raise ValueError(
        f'Cannot normalize unfolding classification with integral {classification_integral}'
    )
classification.Scale(1.0 / classification_integral)
classification.SetTitle(';Pair category;Normalized weighted events')
classification.SetLineColor(ROOT.kRed + 1)
classification.SetMarkerColor(ROOT.kRed + 1)
classification.SetMarkerStyle(20)
classification_positive_values = [
    classification.GetBinContent(bin_index)
    for bin_index in range(1, classification.GetNbinsX() + 1)
    if classification.GetBinContent(bin_index) > 0.0
]
if not classification_positive_values:
    raise ValueError('The unfolding classification has no positive bins for log-y plotting')
classification.SetMinimum(0.5 * min(classification_positive_values))
classification.SetMaximum(10.0 * max(classification_positive_values))
classification.GetXaxis().LabelsOption('v')
classification.GetXaxis().SetLabelSize(0.035)

classification_canvas = ROOT.TCanvas(
    'canvas_unfolding_classification_jerDefExtra',
    'Eta-dependent JER unfolding pair classification', 1000, 750,
)
set_pad_style(classification_canvas, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
classification_canvas.SetBottomMargin(0.30)
classification_canvas.SetLogy(True)
classification.Draw('E1')
classification_labels = []
for y, label in (
    (0.87, training_label),
    (0.82, f'|#eta_{{CM}}^{{jet}}| < {ETA_CUTS[ETA_CUT_INDEX]:g}'),
    (0.77, 'Reco JER def.+#eta-dep.'),
):
    item = ROOT.TLatex(0.20, y, label)
    item.SetNDC(True)
    item.SetTextFont(42)
    item.SetTextSize(0.035)
    item.Draw()
    classification_labels.append(item)
classification_canvas.Modified()
classification_canvas.Update()
save_canvas(
    classification_canvas,
    OUTPUT_DIR / f'{OUTPUT_TAG}_pair_classification.pdf',
    save_png=SAVE_PNG,
)
classification_canvas._classification_objects = [classification, *classification_labels]
classification_canvas


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_unfolding_classification_jerDefExtra
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_pair_classification.pdf has been created


In [59]:
# # For test purpose only, plot the first histogram

# if not genPtEtaCM:
#     raise RuntimeError("genPtEtaCM is empty; load the ROOT file first")

# canvas_name = "canvas_gen_pt_eta_cm_0"
# existing_canvas = ROOT.gROOT.FindObject(canvas_name)
# if existing_canvas:
#     existing_canvas.Close()

# canvas_gen_dijet_pt_eta_cm_0 = ROOT.TCanvas(canvas_name, "cGenDijetPtEtaCM_0", 800, 800)
# canvas_gen_dijet_pt_eta_cm_0.cd()
# set_pad_style(ROOT.gPad, grid_x=False, grid_y=False)
# ROOT.gPad.SetRightMargin(DEFAULT_PLOT_STYLE.palette_right_margin)
# set_2d_style(genPtEtaCM[0])
# genPtEtaCM[0].Draw("COLZ")
# canvas_gen_dijet_pt_eta_cm_0.Modified()
# canvas_gen_dijet_pt_eta_cm_0.Update()
# canvas_gen_dijet_pt_eta_cm_0

In [60]:
# 1D projections of the 2D histograms for each pt_ave bin
genEtaCM = []
recoEtaCM = []
trainingGenEtaCM = []
trainingRecoEtaCM = []
genEtaCMMiss = []
recoEtaCMFake = []

# 2D projections of the THnSparse histograms for each pt_ave bin
gen2recoResponse = []

for i in range(len(eta_cuts)):
    gen_proj = []
    reco_proj = []
    training_gen_proj = []
    training_reco_proj = []
    miss_proj = []
    fake_proj = []
    gen2reco_proj = []

    for j, (low_val, high_val) in enumerate(zip(pt_ave_bins[:-1], pt_ave_bins[1:])):
        gen_low_bin = int(genPtEtaCM[i].GetXaxis().FindBin(low_val + 0.001))
        gen_high_bin = int(genPtEtaCM[i].GetXaxis().FindBin(high_val - 0.001))
        reco_low_bin = int(recoPtEtaCM[i].GetXaxis().FindBin(low_val + 0.001))
        reco_high_bin = int(recoPtEtaCM[i].GetXaxis().FindBin(high_val - 0.001))
        training_gen_low_bin = int(trainingGenPtEtaCM[i].GetXaxis().FindBin(low_val + 0.001))
        training_gen_high_bin = int(trainingGenPtEtaCM[i].GetXaxis().FindBin(high_val - 0.001))
        training_reco_low_bin = int(trainingRecoPtEtaCM[i].GetXaxis().FindBin(low_val + 0.001))
        training_reco_high_bin = int(trainingRecoPtEtaCM[i].GetXaxis().FindBin(high_val - 0.001))
        miss_low_bin = int(genPtEtaCMMiss[i].GetXaxis().FindBin(low_val + 0.001))
        miss_high_bin = int(genPtEtaCMMiss[i].GetXaxis().FindBin(high_val - 0.001))
        fake_low_bin = int(recoPtEtaCMFake[i].GetXaxis().FindBin(low_val + 0.001))
        fake_high_bin = int(recoPtEtaCMFake[i].GetXaxis().FindBin(high_val - 0.001))


        # Project the 2D histograms onto the Y-axis (eta) for the given pt_ave bin
        gen_hist = genPtEtaCM[i].ProjectionY(f"hGenDijetEtaCM_{i}_{j}", gen_low_bin, gen_high_bin)
        reco_hist = recoPtEtaCM[i].ProjectionY(f"hRecoDijetEtaCM_{i}_{j}", reco_low_bin, reco_high_bin)
        training_gen_hist = trainingGenPtEtaCM[i].ProjectionY(f"hTrainingGenDijetEtaCM_{i}_{j}", training_gen_low_bin, training_gen_high_bin)
        training_reco_hist = trainingRecoPtEtaCM[i].ProjectionY(f"hTrainingRecoDijetEtaCM_{i}_{j}", training_reco_low_bin, training_reco_high_bin)
        miss_hist = genPtEtaCMMiss[i].ProjectionY(f"hGenDijetEtaCMMiss_{i}_{j}", miss_low_bin, miss_high_bin)
        fake_hist = recoPtEtaCMFake[i].ProjectionY(f"hRecoDijetEtaCMFake_{i}_{j}", fake_low_bin, fake_high_bin)
        # Set the range for the THnSparse histogram to project onto the Y-axis (eta) for the given pt_ave bin
        response_gen_axis = genPtEtaCMVsRecoPtEtaCM[i].GetAxis(0)
        response_reco_axis = genPtEtaCMVsRecoPtEtaCM[i].GetAxis(2)
        response_gen_axis.SetRange(response_gen_axis.FindBin(low_val + 0.001), response_gen_axis.FindBin(high_val - 0.001))
        response_reco_axis.SetRange(response_reco_axis.FindBin(low_val + 0.001), response_reco_axis.FindBin(high_val - 0.001))
        gen2reco_hist = genPtEtaCMVsRecoPtEtaCM[i].Projection(1, 3)
        gen2reco_hist.SetName(f"hGen2RecoDijetEtaCM_{i}_{j}")

        gen_hist.SetDirectory(0)
        reco_hist.SetDirectory(0)
        training_gen_hist.SetDirectory(0)
        training_reco_hist.SetDirectory(0)
        miss_hist.SetDirectory(0)
        fake_hist.SetDirectory(0)
        gen2reco_hist.SetDirectory(0)

        # gen_hist.Scale(1.0 / gen_hist.Integral() if gen_hist.Integral() > 0 else 1.0)
        # reco_hist.Scale(1.0 / reco_hist.Integral() if reco_hist.Integral() > 0 else 1.0)
        # miss_hist.Scale(1.0 / miss_hist.Integral() if miss_hist.Integral() > 0 else 1.0)
        # fake_hist.Scale(1.0 / fake_hist.Integral() if fake_hist.Integral() > 0 else 1.0)

        gen_proj.append(gen_hist)
        reco_proj.append(reco_hist)
        training_gen_proj.append(training_gen_hist)
        training_reco_proj.append(training_reco_hist)
        miss_proj.append(miss_hist)
        fake_proj.append(fake_hist)
        gen2reco_proj.append(gen2reco_hist)

    genEtaCM.append(gen_proj)
    recoEtaCM.append(reco_proj)
    trainingGenEtaCM.append(training_gen_proj)
    trainingRecoEtaCM.append(training_reco_proj)
    genEtaCMMiss.append(miss_proj)
    recoEtaCMFake.append(fake_proj)
    gen2recoResponse.append(gen2reco_proj)


In [61]:
# # Plot all genEtaCM projections (all pt_ave bins) on one canvas for a given eta cut
# if not genEtaCM:
#     raise RuntimeError("genEtaCM is empty; run the projection cell first")

# eta_idx = 5  # choose eta-cut index here
# if eta_idx < 0 or eta_idx >= len(eta_cuts):
#     raise IndexError(f"eta_idx={eta_idx} is out of range for eta_cuts")

# canvas_name = f"canvas_genEtaCM_allPt_eta{eta_idx}"
# existing_canvas = ROOT.gROOT.FindObject(canvas_name)
# if existing_canvas:
#     existing_canvas.Close()

# canvas_genEtaCM_allPt = ROOT.TCanvas(canvas_name, f"Gen etaCM projections (eta idx {eta_idx})", 800, 800)
# canvas_genEtaCM_allPt.cd()
# set_pad_style(ROOT.gPad, grid_x=False, grid_y=False)

# gen_eta_overlays = []
# max_val = 0.0
# for pt_idx in range(len(pt_ave_bins) - 1):
#     hist = genEtaCM[eta_idx][pt_idx].Clone(f"hGenEtaCM_overlay_eta{eta_idx}_pt{pt_idx}")
#     hist.SetDirectory(0)
#     set_unfolding_1d_style(hist, 'gen')
#     integral = hist.Integral()
#     if integral > 0:
#         hist.Scale(1.0 / integral)
#     hist.SetTitle(";#eta_{CM};Normalized entries")
#     draw_opt = "E1" if pt_idx == 0 else "E1 SAME"
#     hist.Draw(draw_opt)
#     gen_eta_overlays.append(hist)
#     max_val = max(max_val, hist.GetMaximum())

# legend = ROOT.TLegend(0.6, 0.75, 0.88, 0.88)
# set_legend_style(legend)
# legend.SetTextFont(42)
# legend.SetTextSize(0.03)
# for pt_idx in range(len(pt_ave_bins) - 1):
#     label = f"{pt_ave_bins[pt_idx]} < p_{{T}}^{{ave}} < {pt_ave_bins[pt_idx + 1]} GeV"
#     legend.AddEntry(gen_eta_overlays[pt_idx], label, "p")
# legend.Draw()

# text = ROOT.TLatex()
# text.SetNDC(True)
# text.SetTextFont(42)
# text.SetTextSize(0.04)
# text.DrawLatex(0.16, 0.92, f"Gen #eta_{{CM}} projections, eta-cut index = {eta_idx}")

# canvas_genEtaCM_allPt.Modified()
# canvas_genEtaCM_allPt.Update()
# canvas_genEtaCM_allPt

In [62]:
# Plot the 1D projections and response for every pTave interval
if not genEtaCM:
    raise RuntimeError("genEtaCM projections are empty; run the projection cell first")
if not recoEtaCM:
    raise RuntimeError("recoEtaCM projections are empty; run the projection cell first")
if not genEtaCMMiss:
    raise RuntimeError("genEtaCMMiss projections are empty; run the projection cell first")
if not gen2recoResponse:
    raise RuntimeError("gen2recoResponse is empty; run the projection cell first")

eta_idx = ETA_CUT_INDEX
projection_response_canvases = []
projection_response_plot_objects = []

for pt_bin_idx, (pt_low, pt_high) in enumerate(zip(pt_ave_bins[:-1], pt_ave_bins[1:])):
    h_gen_eta = genEtaCM[eta_idx][pt_bin_idx]
    h_reco_eta = recoEtaCM[eta_idx][pt_bin_idx]
    h_miss_eta = genEtaCMMiss[eta_idx][pt_bin_idx]
    h_fake_eta = recoEtaCMFake[eta_idx][pt_bin_idx]
    h_response = gen2recoResponse[eta_idx][pt_bin_idx]
    set_unfolding_1d_style(h_gen_eta, 'gen')
    set_unfolding_1d_style(h_reco_eta, 'reco')
    set_unfolding_1d_style(h_miss_eta, 'miss')
    set_unfolding_1d_style(h_fake_eta, 'fake')

    canvas_name = f'canvas_projection_response_ptBin{pt_bin_idx}'
    existing_canvas = ROOT.gROOT.FindObject(canvas_name)
    if existing_canvas:
        existing_canvas.Close()
    canvas = ROOT.TCanvas(canvas_name, 'Projection and Response', 1400, 600)
    canvas.Divide(2, 1)

    canvas.cd(1)
    set_pad_style(ROOT.gPad, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    h_gen_eta.SetTitle(';#eta_{CM};Entries')
    h_gen_eta.GetXaxis().SetRangeUser(-eta_cuts[eta_idx] - 0.1, eta_cuts[eta_idx] + 0.1)
    h_gen_eta.Draw('E1')
    h_reco_eta.Draw('E1 SAME')
    h_miss_eta.Draw('E1 SAME')
    h_fake_eta.Draw('E1 SAME')
    legend = ROOT.TLegend(0.75, 0.68, 0.88, 0.88)
    set_legend_style(legend)
    legend.AddEntry(h_gen_eta, 'Gen', 'p')
    legend.AddEntry(h_reco_eta, MEASURED_LABEL, 'p')
    legend.AddEntry(h_miss_eta, 'Miss', 'p')
    legend.AddEntry(h_fake_eta, 'Fake', 'p')
    legend.Draw()

    canvas.cd(2)
    set_pad_style(ROOT.gPad, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    ROOT.gPad.SetRightMargin(DEFAULT_PLOT_STYLE.palette_right_margin)
    set_2d_style(h_response)
    h_response.SetTitle(';Reco JER def.+#eta-dep. #eta_{CM};Gen #eta_{CM}')
    h_response.GetXaxis().SetRangeUser(-eta_cuts[eta_idx] - 0.1, eta_cuts[eta_idx] + 0.1)
    h_response.GetYaxis().SetRangeUser(-eta_cuts[eta_idx] - 0.1, eta_cuts[eta_idx] + 0.1)
    h_response.Draw('COLZ')

    labels = []
    for pad_number in (1, 2):
        canvas.cd(pad_number)
        for y, label in (
            (0.85, f'{training_label}; {test_label} spectra'),
            (0.80, f'{pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV'),
            (0.75, 'p_{T}^{Lead} > 50 GeV'),
            (0.70, 'p_{T}^{SubLead} > 40 GeV'),
            (0.65, f'|#eta_{{CM}}| < {eta_cuts[eta_idx]}'),
            (0.60, DIJET_DELTA_PHI_SELECTION_LABEL),
        ):
            item = ROOT.TLatex(0.18, y, label)
            item.SetNDC(True)
            item.SetTextFont(DEFAULT_PLOT_STYLE.font)
            item.SetTextSize(DEFAULT_PLOT_STYLE.annotation_text_size)
            item.Draw()
            labels.append(item)

    canvas.Modified()
    canvas.Update()
    pt_tag = f'pt_{pt_low:g}_{pt_high:g}'
    save_canvas(
        canvas, OUTPUT_DIR / f'{OUTPUT_TAG}_projection_response_{pt_tag}.pdf',
        save_png=SAVE_PNG,
    )
    projection_response_canvases.append(canvas)
    projection_response_plot_objects.append((legend, labels))

projection_response_canvases

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_projection_response_pt_0_40.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_projection_response_pt_40_80.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_projection_response_pt_80_180.pdf has been created


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_projection_response_pt_180_250.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_projection_response_pt_250_300.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_projection_response_pt_300_500.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_projection_response_pt_500_1000.pdf has been created


In [63]:
# Build flattened etaCM histograms and response matrix for one eta-cut
if not genEtaCM or not recoEtaCM or not trainingGenEtaCM or not trainingRecoEtaCM or not genEtaCMMiss or not recoEtaCMFake or not gen2recoResponse:
    raise RuntimeError("Projection containers are empty; run the projection cell first")

eta_idx = ETA_CUT_INDEX
if eta_idx < 0 or eta_idx >= len(eta_cuts):
    raise IndexError(f"eta_idx={eta_idx} is out of range for eta_cuts")

nPtSelections = len(pt_ave_bins) - 1
nEtaBins = genEtaCM[eta_idx][0].GetNbinsX()
nGlobalBins = nPtSelections * nEtaBins

hGenTruthEtaCM = ROOT.TH1D("hGenTruthEtaCM", ";global #eta_{CM} bin;Entries", nGlobalBins, 0.5, nGlobalBins + 0.5)
hTrainingTruthEtaCM = ROOT.TH1D("hTrainingTruthEtaCM", ";global #eta_{CM} bin;Entries", nGlobalBins, 0.5, nGlobalBins + 0.5)
hGenTruthEtaCMMiss = ROOT.TH1D("hGenTruthEtaCMMiss", ";global #eta_{CM} bin;Entries", nGlobalBins, 0.5, nGlobalBins + 0.5)
hRecoMeasuredEtaCM = ROOT.TH1D("hRecoMeasuredEtaCM", ";global #eta_{CM} bin;Entries", nGlobalBins, 0.5, nGlobalBins + 0.5)
hTrainingMeasuredEtaCM = ROOT.TH1D("hTrainingMeasuredEtaCM", ";global #eta_{CM} bin;Entries", nGlobalBins, 0.5, nGlobalBins + 0.5)
hRecoMeasuredEtaCMFake = ROOT.TH1D("hRecoMeasuredEtaCMFake", ";global #eta_{CM} bin;Entries", nGlobalBins, 0.5, nGlobalBins + 0.5)

hResponseEtaCM = ROOT.TH2D(
    "hResponseEtaCM",
    ";global reco #eta_{CM} bin;global gen #eta_{CM} bin",
    nGlobalBins,
    0.5,
    nGlobalBins + 0.5,
    nGlobalBins,
    0.5,
    nGlobalBins + 0.5,
)

for pt_idx in range(nPtSelections):
    h_gen = genEtaCM[eta_idx][pt_idx]
    h_reco = recoEtaCM[eta_idx][pt_idx]
    h_training_gen = trainingGenEtaCM[eta_idx][pt_idx]
    h_training_reco = trainingRecoEtaCM[eta_idx][pt_idx]
    h_miss = genEtaCMMiss[eta_idx][pt_idx]
    h_fake = recoEtaCMFake[eta_idx][pt_idx]

    if any(h.GetNbinsX() != nEtaBins for h in (h_gen, h_reco, h_training_gen, h_training_reco, h_miss, h_fake)):
        raise ValueError("Inconsistent number of eta bins across 1D projections")

    # Map local eta bins to global bins: [pt block][eta bin]
    for eta_bin in range(1, nEtaBins + 1):
        global_bin = pt_idx * nEtaBins + eta_bin

        hGenTruthEtaCM.SetBinContent(global_bin, h_gen.GetBinContent(eta_bin))
        hGenTruthEtaCM.SetBinError(global_bin, h_gen.GetBinError(eta_bin))

        hRecoMeasuredEtaCM.SetBinContent(global_bin, h_reco.GetBinContent(eta_bin))
        hRecoMeasuredEtaCM.SetBinError(global_bin, h_reco.GetBinError(eta_bin))

        hTrainingTruthEtaCM.SetBinContent(global_bin, h_training_gen.GetBinContent(eta_bin))
        hTrainingTruthEtaCM.SetBinError(global_bin, h_training_gen.GetBinError(eta_bin))

        hTrainingMeasuredEtaCM.SetBinContent(global_bin, h_training_reco.GetBinContent(eta_bin))
        hTrainingMeasuredEtaCM.SetBinError(global_bin, h_training_reco.GetBinError(eta_bin))

        hGenTruthEtaCMMiss.SetBinContent(global_bin, h_miss.GetBinContent(eta_bin))
        hGenTruthEtaCMMiss.SetBinError(global_bin, h_miss.GetBinError(eta_bin))

        hRecoMeasuredEtaCMFake.SetBinContent(global_bin, h_fake.GetBinContent(eta_bin))
        hRecoMeasuredEtaCMFake.SetBinError(global_bin, h_fake.GetBinError(eta_bin))

# Fill flattened response matrix including pt-migration off-diagonal blocks
sparse_resp = genPtEtaCMVsRecoPtEtaCM[eta_idx]
for gen_pt_idx, (gen_low, gen_high) in enumerate(zip(pt_ave_bins[:-1], pt_ave_bins[1:])):
    gen_response_axis = sparse_resp.GetAxis(0)
    gen_low_bin = int(gen_response_axis.FindBin(gen_low + 0.001))
    gen_high_bin = int(gen_response_axis.FindBin(gen_high - 0.001))

    sparse_resp.GetAxis(0).SetRange(gen_low_bin, gen_high_bin)

    for reco_pt_idx, (reco_low, reco_high) in enumerate(zip(pt_ave_bins[:-1], pt_ave_bins[1:])):
        reco_response_axis = sparse_resp.GetAxis(2)
        reco_low_bin = int(reco_response_axis.FindBin(reco_low + 0.001))
        reco_high_bin = int(reco_response_axis.FindBin(reco_high - 0.001))

        sparse_resp.GetAxis(2).SetRange(reco_low_bin, reco_high_bin)
        h_resp_block = sparse_resp.Projection(1, 3)
        h_resp_block.SetName(f"hResponseBlock_genPt{gen_pt_idx}_recoPt{reco_pt_idx}")

        for gen_eta_bin in range(1, nEtaBins + 1):
            global_gen_bin = gen_pt_idx * nEtaBins + gen_eta_bin
            for reco_eta_bin in range(1, nEtaBins + 1):
                global_reco_bin = reco_pt_idx * nEtaBins + reco_eta_bin

                content = h_resp_block.GetBinContent(reco_eta_bin, gen_eta_bin)
                error = h_resp_block.GetBinError(reco_eta_bin, gen_eta_bin)

                hResponseEtaCM.SetBinContent(global_reco_bin, global_gen_bin, content)
                hResponseEtaCM.SetBinError(global_reco_bin, global_gen_bin, error)

# Reset sparse axis ranges to full range
sparse_resp.GetAxis(0).SetRange(0, -1)
sparse_resp.GetAxis(2).SetRange(0, -1)

hGenTruthEtaCM.SetDirectory(0)
hTrainingTruthEtaCM.SetDirectory(0)
hGenTruthEtaCMMiss.SetDirectory(0)
hRecoMeasuredEtaCM.SetDirectory(0)
hTrainingMeasuredEtaCM.SetDirectory(0)
hRecoMeasuredEtaCMFake.SetDirectory(0)
hResponseEtaCM.SetDirectory(0)

print(f"Built flattened histograms for eta_idx={eta_idx}")
print(f"nPtSelections={nPtSelections}, nEtaBins={nEtaBins}, nGlobalBins={nGlobalBins}")

Built flattened histograms for eta_idx=5
nPtSelections=7, nEtaBins=72, nGlobalBins=504


In [64]:
# Plot 1D flattened histograms and 2D response matrix for test purpose

canvas_name = "canvas_flattened"
existing_canvas = ROOT.gROOT.FindObject(canvas_name)
if existing_canvas:
    existing_canvas.Close()

canvas_flattened = ROOT.TCanvas("canvas_flattened", "Flattened Histograms and Response", 1400, 600)
canvas_flattened.Divide(2, 1)

canvas_flattened.cd(1)
set_pad_style(ROOT.gPad, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
set_unfolding_1d_style(hGenTruthEtaCM, 'gen')
set_unfolding_1d_style(hRecoMeasuredEtaCM, 'reco')
set_unfolding_1d_style(hGenTruthEtaCMMiss, 'miss')
set_unfolding_1d_style(hRecoMeasuredEtaCMFake, 'fake')
hGenTruthEtaCM.SetTitle(";#eta_{CM} bin;Entries")
hGenTruthEtaCM.Draw("E")
hRecoMeasuredEtaCM.Draw("E SAME")
hGenTruthEtaCMMiss.Draw("E SAME")
hRecoMeasuredEtaCMFake.Draw("E SAME")

legend_flat = ROOT.TLegend(0.55, 0.68, 0.88, 0.88)
set_legend_style(legend_flat)
legend_flat.AddEntry(hGenTruthEtaCM, f"{TEST_DIRECTION} Gen", "p")
legend_flat.AddEntry(hRecoMeasuredEtaCM, f"{TEST_DIRECTION} {MEASURED_LABEL}", "p")
legend_flat.AddEntry(hGenTruthEtaCMMiss, f"{TRAIN_DIRECTION} Miss", "p")
legend_flat.AddEntry(hRecoMeasuredEtaCMFake, f"{TRAIN_DIRECTION} Fakes", "p")
legend_flat.Draw()

text.DrawLatexNDC(0.18, 0.85, f"{test_label}")
text.DrawLatexNDC(0.45, 0.92, f"|#eta_{{CM}}| < {eta_cuts[eta_idx]}")

canvas_flattened.cd(2)
set_pad_style(ROOT.gPad, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
ROOT.gPad.SetRightMargin(DEFAULT_PLOT_STYLE.palette_right_margin)
set_2d_style(hResponseEtaCM)
hResponseEtaCM.SetTitle(";Reco JER def.+#eta-dep. #eta_{CM} bin;Gen #eta_{CM} bin")
hResponseEtaCM.Draw("COLZ")
text.DrawLatexNDC(0.45, 0.92, f"|#eta_{{CM}}| < {eta_cuts[eta_idx]}")

canvas_flattened.Modified()
canvas_flattened.Update()
save_canvas(
    canvas_flattened, OUTPUT_DIR / f'{OUTPUT_TAG}_flattened_response.pdf', save_png=SAVE_PNG
)
canvas_flattened

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_flattened_response.pdf has been created


In [65]:
# Prepare and run the unfolding using RooUnfold
if not hResponseEtaCM or not hRecoMeasuredEtaCM:
    raise RuntimeError("Response matrix or measured histogram is empty; run the previous cell first")
if not hGenTruthEtaCM or not hTrainingTruthEtaCM or not hTrainingMeasuredEtaCM:
    raise RuntimeError("Test truth or training marginals are empty; run the previous cell first")
if not hGenTruthEtaCMMiss:
    raise RuntimeError("Missed truth histogram is empty; run the previous cell first")
if not hRecoMeasuredEtaCMFake:
    raise RuntimeError("Fake measured histogram is empty; run the previous cell first")

# The target-range response contains matched entries with both truth and reco
# inside the configured pT bins. Residuals of the full truth/measured spectra
# relative to its projections are the effective misses/fakes; they include both
# explicit producer categories and migrations across the target pT boundaries.
hMatchedRecoEtaCM = hResponseEtaCM.ProjectionX("hMatchedRecoEtaCM")
hMatchedTruthEtaCM = hResponseEtaCM.ProjectionY("hMatchedTruthEtaCM")
hMatchedRecoEtaCM.SetDirectory(0)
hMatchedTruthEtaCM.SetDirectory(0)

hEffectiveMissEtaCM = hTrainingTruthEtaCM.Clone("hEffectiveMissEtaCM")
hEffectiveMissEtaCM.Add(hMatchedTruthEtaCM, -1.0)
hEffectiveFakeEtaCM = hTrainingMeasuredEtaCM.Clone("hEffectiveFakeEtaCM")
hEffectiveFakeEtaCM.Add(hMatchedRecoEtaCM, -1.0)

hBoundaryMissEtaCM = hEffectiveMissEtaCM.Clone("hBoundaryMissEtaCM")
hBoundaryMissEtaCM.Add(hGenTruthEtaCMMiss, -1.0)
hBoundaryFakeEtaCM = hEffectiveFakeEtaCM.Clone("hBoundaryFakeEtaCM")
hBoundaryFakeEtaCM.Add(hRecoMeasuredEtaCMFake, -1.0)
for hist in (hEffectiveMissEtaCM, hEffectiveFakeEtaCM, hBoundaryMissEtaCM, hBoundaryFakeEtaCM):
    hist.SetDirectory(0)

def assert_no_negative_bins(hist, label, tolerance=1.0e-9):
    negative_bins = [
        i for i in range(1, hist.GetNbinsX() + 1)
        if hist.GetBinContent(i) < -tolerance
    ]
    if negative_bins:
        raise ValueError(f"{label} has negative contents in bins {negative_bins[:10]}")

assert_no_negative_bins(hEffectiveMissEtaCM, "Effective miss")
assert_no_negative_bins(hEffectiveFakeEtaCM, "Effective fake")
assert_no_negative_bins(hBoundaryMissEtaCM, "Boundary-migration miss")
assert_no_negative_bins(hBoundaryFakeEtaCM, "Boundary-migration fake")

# Scale common clones before constructing RooUnfoldResponse. RooUnfold sanitizes
# square response matrices at an absolute 1e-9 threshold, which otherwise
# corrupts small weighted high-pT/large-|eta| response columns.
hTrainingRecoForResponse = hTrainingMeasuredEtaCM.Clone('hTrainingRecoForResponse')
hTrainingTruthForResponse = hTrainingTruthEtaCM.Clone('hTrainingTruthForResponse')
hTestRecoForUnfold = hRecoMeasuredEtaCM.Clone('hTestRecoForUnfold')
hResponseForUnfold = hResponseEtaCM.Clone('hResponseForUnfold')
for hist in (hTrainingRecoForResponse, hTrainingTruthForResponse, hTestRecoForUnfold, hResponseForUnfold):
    hist.SetDirectory(0)
    hist.Scale(RESPONSE_SCALE)

# This constructor derives inefficiency from scaled truth minus the scaled
# response truth projection, and likewise derives scaled fakes. Do not call
# response.Miss/Fake afterwards; that would double count them.
response = ROOT.RooUnfoldResponse(
    hTrainingRecoForResponse,
    hTrainingTruthForResponse,
    hResponseForUnfold,
    "responseEtaCM",
    "Flattened dijet pTave-etaCM response",
)

response_fakes = response.Hfakes()
for bin_idx in range(1, nGlobalBins + 1):
    expected = RESPONSE_SCALE * hEffectiveFakeEtaCM.GetBinContent(bin_idx)
    actual = response_fakes.GetBinContent(bin_idx)
    tolerance = 1.0e-9 * max(1.0, abs(expected), abs(actual))
    if abs(actual - expected) > tolerance:
        raise ValueError(
            f"RooUnfold fake mismatch in global bin {bin_idx}: "
            f"expected {expected}, got {actual}"
        )

print(f"Explicit misses: {hGenTruthEtaCMMiss.Integral():.6g}")
print(f"Boundary misses: {hBoundaryMissEtaCM.Integral():.6g}")
print(f"Effective misses: {hEffectiveMissEtaCM.Integral():.6g}")
print(f"Explicit fakes: {hRecoMeasuredEtaCMFake.Integral():.6g}")
print(f"Boundary fakes: {hBoundaryFakeEtaCM.Integral():.6g}")
print(f"Effective fakes: {hEffectiveFakeEtaCM.Integral():.6g}")

# Explicitly enable fake handling: this RooUnfold version defaults to False.
# Misses are already represented by the response inefficiency.
if not response.HasFakes():
    raise RuntimeError("The unfolding response does not contain the expected fake component")
unfold = ROOT.RooUnfoldBayes(
    response, hTestRecoForUnfold, N_ITERATIONS, False, True
)
unfold.SetVerbose(-1)
hUnfoldedEtaCM = unfold.Hunfold()
hUnfoldedEtaCM.Scale(1.0 / RESPONSE_SCALE)
hUnfoldedEtaCM.SetName("hUnfoldedEtaCM")
hUnfoldedEtaCM.SetTitle(";#eta_{CM} bin;Entries")
hUnfoldedEtaCM.SetDirectory(0)

# Compute the covariance matrix of the unfolded result.
# RooUnfold 3.x exposes Eunfold rather than the legacy Ereco method.
covariance_matrix = unfold.Eunfold(ROOT.RooUnfolding.kCovariance)
covariance_matrix *= 1.0 / (RESPONSE_SCALE * RESPONSE_SCALE)

Explicit misses: 0.000578528
Boundary misses: 1.10897e-08
Effective misses: 0.000578539
Explicit fakes: 0.000732519
Boundary fakes: 5.66639e-11
Effective fakes: 0.000732519
An additional truth bin is added to handle 7.32519e+08 fakes.


In [66]:
# Plot the comparison of the unfolded result with the truth and measured histograms on one histogram
# and the ratio of unfolded and measured to truth on another histogram
canvas_name = "canvas_unfolded"
existing_canvas = ROOT.gROOT.FindObject(canvas_name)
if existing_canvas:
    existing_canvas.Close()

canvas_unfolded = ROOT.TCanvas(canvas_name, "Unfolded Result Comparison", 1400, 600)
canvas_unfolded.Divide(2, 1)

canvas_unfolded.cd(1)
set_pad_style(ROOT.gPad, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
set_unfolding_1d_style(hGenTruthEtaCM, 'gen')
set_unfolding_1d_style(hRecoMeasuredEtaCM, 'reco')
set_unfolding_1d_style(hUnfoldedEtaCM, 'unfolded')
hGenTruthEtaCM.Draw("E")
hRecoMeasuredEtaCM.Draw("E SAME")
hUnfoldedEtaCM.Draw("E SAME")

legend_unfolded = ROOT.TLegend(0.6, 0.68, 0.88, 0.88)
set_legend_style(legend_unfolded)
legend_unfolded.AddEntry(hGenTruthEtaCM, "Gen", "p")
legend_unfolded.AddEntry(hRecoMeasuredEtaCM, MEASURED_LABEL, "p")
legend_unfolded.AddEntry(hUnfoldedEtaCM, f"Unfolded", "p")
legend_unfolded.Draw()

text.DrawLatexNDC(0.18, 0.85, f"{test_label}")
text.DrawLatexNDC(0.45, 0.92, f"|#eta_{{CM}}| < {eta_cuts[eta_idx]}")

canvas_unfolded.cd(2)
set_pad_style(ROOT.gPad, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
hMeasuredToTruth = hRecoMeasuredEtaCM.Clone("hMeasuredToTruth")
hMeasuredToTruth.Divide(hGenTruthEtaCM)
hUnfoldedToTruth = hUnfoldedEtaCM.Clone("hUnfoldedToTruth")
hUnfoldedToTruth.Divide(hGenTruthEtaCM)
set_unfolding_1d_style(hMeasuredToTruth, 'reco')
set_unfolding_1d_style(hUnfoldedToTruth, 'unfolded')

hMeasuredToTruth.SetTitle(";#eta_{CM} bin;Ratio to Gen")
hMeasuredToTruth.Draw("E")
hUnfoldedToTruth.Draw("E SAME")
hMeasuredToTruth.GetYaxis().SetRangeUser(*FLATTENED_RATIO_TO_GEN_Y_RANGE)

legend_ratio = ROOT.TLegend(0.6, 0.68, 0.88, 0.88)
set_legend_style(legend_ratio)
legend_ratio.AddEntry(hMeasuredToTruth, "Reco JER def.+#eta-dep. / Gen", "p")
legend_ratio.AddEntry(hUnfoldedToTruth, f"Unfolded / Gen", "p")
legend_ratio.Draw()

text.DrawLatexNDC(0.18, 0.85, f"{test_label}")
text.DrawLatexNDC(0.45, 0.92, f"|#eta_{{CM}}| < {eta_cuts[eta_idx]}")

canvas_unfolded.Modified()
canvas_unfolded.Update()
save_canvas(
    canvas_unfolded, OUTPUT_DIR / f'{OUTPUT_TAG}_closure.pdf', save_png=SAVE_PNG
)
canvas_unfolded

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_closure.pdf has been created


## Unfolded eta distributions in all pTave intervals

Extract every pTave block from the flattened unfolded histogram. For each interval, compare unfolded, gen, and eta-dependent JER-default reco eta distributions and plot reco/gen and unfolded/gen ratios.

In [67]:
hGenEtaCMByPt = []
hRecoEtaCMByPt = []
hUnfoldedEtaCMByPt = []
hRecoToGenEtaCMByPt = []
hUnfoldedToGenEtaCMByPt = []
unfolded_eta_canvases = []
unfolded_eta_plot_objects = []

for selected_pt_idx, (selected_pt_low, selected_pt_high) in enumerate(
    zip(pt_ave_bins[:-1], pt_ave_bins[1:])
):
    selected_pt_tag = f'pt_{selected_pt_low:g}_{selected_pt_high:g}'
    h_gen = genEtaCM[eta_idx][selected_pt_idx].Clone(f'hGenEtaCM_ptBin{selected_pt_idx}')
    h_reco = recoEtaCM[eta_idx][selected_pt_idx].Clone(f'hRecoEtaCM_ptBin{selected_pt_idx}')
    h_unfolded = h_gen.Clone(f'hUnfoldedEtaCM_ptBin{selected_pt_idx}')
    h_unfolded.Reset('ICES')
    for eta_bin in range(1, nEtaBins + 1):
        global_bin = selected_pt_idx * nEtaBins + eta_bin
        h_unfolded.SetBinContent(eta_bin, hUnfoldedEtaCM.GetBinContent(global_bin))
        h_unfolded.SetBinError(eta_bin, hUnfoldedEtaCM.GetBinError(global_bin))

    for hist in (h_gen, h_reco, h_unfolded):
        hist.SetDirectory(0)
    set_unfolding_1d_style(h_gen, 'gen')
    set_unfolding_1d_style(h_reco, 'reco')
    set_unfolding_1d_style(h_unfolded, 'unfolded')

    h_reco_ratio = h_reco.Clone(f'hRecoToGenEtaCM_ptBin{selected_pt_idx}')
    h_reco_ratio.Divide(h_gen)
    h_unfolded_ratio = h_unfolded.Clone(f'hUnfoldedToGenEtaCM_ptBin{selected_pt_idx}')
    h_unfolded_ratio.Divide(h_gen)
    set_unfolding_1d_style(h_reco_ratio, 'reco')
    set_unfolding_1d_style(h_unfolded_ratio, 'unfolded')

    canvas_name = f'canvas_unfolded_ptBin{selected_pt_idx}'
    existing_canvas = ROOT.gROOT.FindObject(canvas_name)
    if existing_canvas:
        existing_canvas.Close()
    canvas = ROOT.TCanvas(canvas_name, 'Unfolded eta distribution by pTave', 800, 800)
    top = ROOT.TPad(f'{canvas_name}_top', '', 0.0, 0.30, 1.0, 1.0)
    bottom = ROOT.TPad(f'{canvas_name}_bottom', '', 0.0, 0.0, 1.0, 0.30)
    top.Draw()
    bottom.Draw()

    top.cd()
    set_pad_style(top, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    top.SetBottomMargin(DEFAULT_PLOT_STYLE.ratio_top_bottom_margin)
    maximum = max(h_gen.GetMaximum(), h_reco.GetMaximum(), h_unfolded.GetMaximum())
    h_gen.SetMaximum(1.30 * maximum if maximum > 0.0 else 1.0)
    h_gen.SetTitle(';#eta_{CM};Entries')
    h_gen.GetXaxis().SetRangeUser(-eta_cuts[eta_idx] - 0.1, eta_cuts[eta_idx] + 0.1)
    h_gen.GetXaxis().SetLabelSize(0.0)
    h_gen.Draw('E1')
    h_reco.Draw('E1 SAME')
    h_unfolded.Draw('E1 SAME')
    legend = ROOT.TLegend(0.60, 0.70, 0.88, 0.88)
    set_legend_style(legend)
    legend.AddEntry(h_gen, 'Gen', 'p')
    legend.AddEntry(h_reco, MEASURED_LABEL, 'p')
    legend.AddEntry(h_unfolded, 'Unfolded', 'p')
    legend.Draw()
    labels = []
    for y, label in (
        (0.86, f'{training_label}; {test_label} spectra'),
        (0.81, f'{selected_pt_low:g} < p_{{T}}^{{ave}} < {selected_pt_high:g} GeV'),
        (0.76, f'|#eta_{{CM}}| < {eta_cuts[eta_idx]}'),
    ):
        item = ROOT.TLatex(0.18, y, label)
        item.SetNDC(True)
        item.SetTextFont(DEFAULT_PLOT_STYLE.font)
        item.SetTextSize(DEFAULT_PLOT_STYLE.annotation_text_size)
        item.Draw()
        labels.append(item)

    bottom.cd()
    set_pad_style(bottom, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    bottom.SetTopMargin(DEFAULT_PLOT_STYLE.ratio_bottom_top_margin)
    bottom.SetBottomMargin(DEFAULT_PLOT_STYLE.ratio_bottom_margin)
    for ratio in (h_reco_ratio, h_unfolded_ratio):
        ratio.GetXaxis().SetRangeUser(-eta_cuts[eta_idx] - 0.1, eta_cuts[eta_idx] + 0.1)
        ratio.GetXaxis().SetTitleSize(DEFAULT_PLOT_STYLE.ratio_axis_title_size)
        ratio.GetXaxis().SetLabelSize(DEFAULT_PLOT_STYLE.ratio_axis_label_size)
        ratio.GetYaxis().SetTitleSize(DEFAULT_PLOT_STYLE.ratio_axis_title_size)
        ratio.GetYaxis().SetLabelSize(DEFAULT_PLOT_STYLE.ratio_axis_label_size)
        ratio.GetYaxis().SetTitleOffset(DEFAULT_PLOT_STYLE.ratio_y_title_offset)
        ratio.GetYaxis().SetNdivisions(DEFAULT_PLOT_STYLE.ratio_axis_divisions)
    h_reco_ratio.SetTitle(';#eta_{CM};Ratio to Gen')
    h_reco_ratio.GetYaxis().SetRangeUser(*ETA_RATIO_TO_GEN_Y_RANGE)
    h_reco_ratio.Draw('E1')
    h_unfolded_ratio.Draw('E1 SAME')

    canvas.Modified()
    canvas.Update()
    save_canvas(
        canvas, OUTPUT_DIR / f'{OUTPUT_TAG}_eta_overlay_ratio_{selected_pt_tag}.pdf',
        save_png=SAVE_PNG,
    )
    hGenEtaCMByPt.append(h_gen)
    hRecoEtaCMByPt.append(h_reco)
    hUnfoldedEtaCMByPt.append(h_unfolded)
    hRecoToGenEtaCMByPt.append(h_reco_ratio)
    hUnfoldedToGenEtaCMByPt.append(h_unfolded_ratio)
    unfolded_eta_canvases.append(canvas)
    unfolded_eta_plot_objects.append((top, bottom, legend, labels))

unfolded_eta_canvases

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_eta_overlay_ratio_pt_0_40.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_eta_overlay_ratio_pt_40_80.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_eta_overlay_ratio_pt_80_180.pdf has been created


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_eta_overlay_ratio_pt_180_250.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_eta_overlay_ratio_pt_250_300.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_eta_overlay_ratio_pt_300_500.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6_eta_overlay_ratio_pt_500_1000.pdf has been created


## Save unfolding output

Write the flattened spectra, response diagnostics, unfolded result, ratios, covariance matrix, and configuration metadata to `hist_analysis/output/unfold2D/`.

In [68]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_file = ROOT.TFile.Open(str(OUTPUT_ROOT_FILE), 'RECREATE')
if not output_file or output_file.IsZombie():
    raise OSError(f'Unable to create unfolding output file: {OUTPUT_ROOT_FILE}')

output_histograms = (
    hGenTruthEtaCM,
    hRecoMeasuredEtaCM,
    hTrainingTruthEtaCM,
    hTrainingMeasuredEtaCM,
    hGenTruthEtaCMMiss,
    hRecoMeasuredEtaCMFake,
    hResponseEtaCM,
    hMatchedTruthEtaCM,
    hMatchedRecoEtaCM,
    hEffectiveMissEtaCM,
    hEffectiveFakeEtaCM,
    hBoundaryMissEtaCM,
    hBoundaryFakeEtaCM,
    hUnfoldedEtaCM,
    hMeasuredToTruth,
    hUnfoldedToTruth,
    *hGenEtaCMByPt,
    *hRecoEtaCMByPt,
    *hUnfoldedEtaCMByPt,
    *hRecoToGenEtaCMByPt,
    *hUnfoldedToGenEtaCMByPt,
)

try:
    output_file.cd()
    for hist in output_histograms:
        if hist.Write() <= 0:
            raise OSError(f'Failed to write {hist.GetName()} to {OUTPUT_ROOT_FILE}')
    covariance_matrix.Write('hUnfoldedCovariance')
    response.Write('responseEtaCM')
    configuration = ROOT.TNamed(
        'unfoldingConfiguration',
        (f'generator={GENERATOR};train_direction={TRAIN_DIRECTION};test_direction={TEST_DIRECTION};'
         f'train_input={train_input_path};test_input={test_input_path};eta_cut={eta_cuts[eta_idx]:g};'
         f'pt_ave_bins={list(pt_ave_bins)};iterations={N_ITERATIONS};'
         f'flattened_ratio_to_gen_y_range={FLATTENED_RATIO_TO_GEN_Y_RANGE};'
         f'eta_ratio_to_gen_y_range={ETA_RATIO_TO_GEN_Y_RANGE};'
         f'draw_grid={DRAW_GRID};'
         f'response_scale={RESPONSE_SCALE:g};measured_histogram={MEASURED_HISTOGRAM_TEMPLATE};'
         f'response_histogram={RESPONSE_HISTOGRAM_TEMPLATE};miss_histogram={MISS_HISTOGRAM_TEMPLATE};'
         f'fake_histogram={FAKE_HISTOGRAM_TEMPLATE};classification_histogram={CLASSIFICATION_HISTOGRAM_TEMPLATE};'
         'handle_fakes=true'),
    )
    configuration.Write()
finally:
    output_file.Close()

print(f'Wrote unfolding output to {OUTPUT_ROOT_FILE}')

Wrote unfolding output to /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D/embedding_Pbgoing_response_pgoing_test_unfold2D_jerDefExtra_eta_1p9_iter_6.root
